# Bollinger Band Reversal (BBR)

## Import Libs

In [3]:
import pandas as pd 
import numpy as np
import os
from pandas import DataFrame, Series
import plotly.graph_objects as go

## Functions

### Get FE Data

In [4]:
def get_fe_price_data(
        filename: str = "FE_V2_GBPUSD_15mins_1yr_End_20250311.csv"
        ) -> DataFrame:
    """
    Return the FE price data as a DatetimeIndexed 
    DataFrame set to US/Eastern TZ
    """
    FOLDER = "price_data"
    PATH = f"{os.getcwd()}/{FOLDER}"
    df = pd.read_csv(f"{PATH}/{filename}")
    df["Date"] = pd.DatetimeIndex(df["Date"], tz="US/Eastern")
    df.set_index("Date", inplace=True)
    return df

In [5]:
# Set new columns 
gains_cols = [ 
    "Win", "Loss", 
    "TP", "SL", 
    "Gain",
    "Trade_Start", "Trade_End"
    ]

### Simulate Short Positions (Range-Based)

In [695]:
def range_short_gains(
        df: Series, 
        high: Series, 
        low: Series,
        close: Series,
        signal_name: str,
        sl_pct_range: int,
        tp_pct_range: int,
        bbl: Series,
        atr4: Series,
        range_type: str = "ADR",
        momentum_trade_mgmt = True
        ):
    """Get the pip gain and apply to df"""

    win = 0
    loss = 0
    target_pips = 0 
    sl_pips = 0
    gain = 0
    trade_start =  None
    trade_end = None
    
    if df[signal_name] is True:
        idx = int(df["Idx"])
        iday_idx = int(df["Iday_Idx"])
        # get signal start of fx day timestamp
        day_start_ts = high.iloc[idx-iday_idx:idx-iday_idx+1].index[0]
        OFFSET = day_start_ts + pd.DateOffset(days=1) # end of signal's fx day
        START = df.name # signal start time
        TD = pd.Timedelta(minutes=15)
        END = OFFSET - TD # 17:00 on the signals fx day

        # get stop loss time:
        sl_window = high.loc[START+TD:END] # from signal idx+1 to EOD
        close_window = close.loc[START+TD:END] # from signal idx+1 to EOD
        bbl_window = bbl.loc[START+TD:END]
        atr4_window = atr4.loc[START+TD:END]
        stop = False # update if stopped 
        sl_ts = None # get timestamp when stopped out
        momentum_stop = False

        for i in range(len(sl_window)): 
            stop_condition = sl_window.iloc[i] >= (df["Close"] + (df[range_type] * sl_pct_range))
            momentum_condition = close_window.iloc[i] < bbl_window.iloc[i] if momentum_trade_mgmt is True else False
            if stop_condition:
                stop = True # trade hit stop loss
                # get stop loss timestamp
                sl_ts = sl_window.iloc[i:i+1].index[0] 
                break
            if momentum_condition:
                momentum_stop = True
                sl_ts = close_window.iloc[i:i+1].index[0]
                break

        # if not stopped out then the stop window is signal:EOD
        # don't include trade if it is the last candle of the day
        if stop is False and momentum_stop is False:
            if close_window.empty is False:
                sl_ts = close_window.iloc[-1:].index[0]
            else:
                sl_ts = START+TD
        # Take profit price
        tp = df["Close"] - (df[range_type] * tp_pct_range)
        
        # trade window
        tp_window = low[START+TD:sl_ts+TD]
        trade_start = START+TD
        trade_end = sl_ts
        target_pips = df["Close"] - tp
        sl_pips = df["Close"] - (df["Close"] + df[range_type] * sl_pct_range) 
        if tp_window.min() <= tp:
            if trade_start == trade_end:
                if stop is False and momentum_stop is False:
                    win = 1
                    gain = df["Close"] - tp
                elif momentum_stop is True:
                    gain = df["Close"] - close.loc[sl_ts]
                    if gain > 0:
                        win = 1
                    else:
                        loss = 1
                else:
                    loss = 1
                    gain = sl_pips
            else:
                win = 1
                gain = df["Close"] - tp                                
        else:
            if stop is True:
                loss = 1
                gain = sl_pips
            elif momentum_stop is True:
                gain = df["Close"] - close.loc[sl_ts]
                if gain > 0:
                    win = 1
                else:
                    loss = 1
            else:
                gain = df["Close"] - close_window.iloc[-1] if close_window.empty is False else 0
                if gain > 0:
                    win = 1
                else:
                    loss = 1
    
    data = win, loss, \
        target_pips, sl_pips, \
        gain, \
        trade_start, trade_end
    
    return data


### Simulate Long Positions (Range-Based)

In [694]:
def range_long_gains(
        df: Series, 
        high: Series, 
        low: Series,
        close: Series,
        signal_name: str,
        sl_pct_range: int,
        tp_pct_range: int,
        bbu: Series,
        atr4: Series,
        range_type: str = "ADR",
        momentum_trade_mgmt = True
        ):
    """Get the pip gain and apply to df"""

    win = 0
    loss = 0
    target_pips = 0 
    sl_pips = 0
    gain = 0
    trade_start =  None
    trade_end = None
    
    if df[signal_name] is True:
        idx = int(df["Idx"])
        iday_idx = int(df["Iday_Idx"])
        # get signal start of fx day timestamp
        day_start_ts = low.iloc[idx-iday_idx:idx-iday_idx+1].index[0]
        OFFSET = day_start_ts + pd.DateOffset(days=1) # end of signal's fx day
        START = df.name # signal start time
        TD = pd.Timedelta(minutes=15)
        END = OFFSET - TD # 17:00 on the signals fx day

        # get stop loss time:
        sl_window = low.loc[START+TD:END] # from signal idx+1 to EOD
        close_window = close.loc[START+TD:END] # from signal idx+1 to EOD
        bbu_window = bbu.loc[START+TD:END]
        atr4_window = atr4.loc[START+TD:END]
        stop = False # update if stopped 
        sl_ts = None # get timestamp when stopped out
        momentum_stop = False

        for i in range(len(sl_window)):
            stop_condition = sl_window.iloc[i] <= (df["Close"] - df[range_type] * sl_pct_range)
            momentum_condition = close_window.iloc[i] > bbu_window.iloc[i] if momentum_trade_mgmt is True else False
            if stop_condition:
                stop = True # trade hit stop loss
                # get stop loss timestamp
                sl_ts = sl_window.iloc[i:i+1].index[0] 
                break
            if momentum_condition:
                momentum_stop = True
                sl_ts = close_window.iloc[i:i+1].index[0]
                break
        # if not stopped out then the stop window is signal:EOD
        # don't include trade if it is the last candle of the day
        if stop is False and momentum_stop is False:
            if close_window.empty is False:
                sl_ts = close_window.iloc[-1:].index[0]
            else:
                sl_ts = START+TD

        # Take profit price
        tp = df["Close"] + (df[range_type] * tp_pct_range)
           
        # trade window
        tp_window = high[START+TD:sl_ts+TD]
        trade_start = START+TD
        trade_end = sl_ts
        target_pips = tp - df["Close"]
        sl_pips = (df["Close"] - df[range_type] * sl_pct_range) - df["Close"] 
        if tp_window.max() >= tp:
            if trade_start == trade_end:
                if stop is False and momentum_stop is False:
                    win = 1
                    gain = tp - df["Close"]
                elif momentum_stop is True:
                    gain = close.loc[sl_ts] - df["Close"]
                    if gain > 0:
                        win = 1
                    else:
                        loss = 1
                else:
                    loss = 1
                    gain = sl_pips
            else:
                win = 1
                gain = tp - df["Close"]
        else:
            if stop is True:
                loss = 1
                gain = sl_pips
            elif momentum_stop is True:
                gain = close.loc[sl_ts] - df["Close"]
                if gain > 0:
                    win = 1
                else:
                    loss = 1
            else:
                gain = close_window.iloc[-1] - df["Close"] if close_window.empty is False else 0
                if gain > 0:
                    win = 1
                else:
                    loss = 1
    
    data = win, loss, \
        target_pips, sl_pips, \
        gain, \
        trade_start, trade_end, \
        
    return data


### Simulate Limit Short Position

In [416]:
def limit_short_gains(
        df: Series, 
        high: Series, 
        low: Series,
        close: Series,
        signal_name: str,
        sl_pct_range: int,
        tp_pct_range: int,
        limit_entry: Series,
        range_type: str = "ADR",
        s_r: bool = False
        ):
    """Get the pip gain and apply to df"""

    win = 0
    loss = 0
    target_pips = 0 
    sl_pips = 0
    gain = 0
    trade_start =  None
    trade_end = None

    signal = None
    if s_r is True:
        if df["S_R_Signal"] in [1]:
            signal = True
        elif df["S_R_Signal"] in [2,3,4]:
            signal = False
    else:
        signal = df[signal_name]
    
    if signal is True:
        idx = int(df["Idx"])
        iday_idx = int(df["Iday_Idx"])
        entry = limit_entry.iloc[idx] 
        # get signal start of fx day timestamp
        day_start_ts = high.iloc[idx-iday_idx:idx-iday_idx+1].index[0]
        OFFSET = day_start_ts + pd.DateOffset(days=1) # end of signal's fx day
        START = df.name # signal start time
        TD = pd.Timedelta(minutes=15)
        END = OFFSET - TD # 17:00 on the signals fx day

        # get stop loss time:
        sl_window = high.loc[START+TD:END] # from signal idx+1 to EOD
        close_window = close.loc[START+TD:END] # from signal idx+1 to EOD
        stop = False # update if stopped 
        sl_ts = None # get timestamp when stopped out

        for i in range(len(sl_window)): 
            stop_condition = sl_window.iloc[i] >= (entry + (df[range_type] * sl_pct_range))
            if stop_condition:
                stop = True # trade hit stop loss
                # get stop loss timestamp
                sl_ts = sl_window.iloc[i:i+1].index[0] 
                break

        # if not stopped out then the stop window is signal:EOD
        # don't include trade if it is the last candle of the day
        if stop is False:
            if close_window.empty is False:
                sl_ts = close_window.iloc[-1:].index[0]
            else:
                sl_ts = START+TD
        # Take profit price
        tp = entry - (df[range_type] * tp_pct_range)
        
        # trade window
        tp_window = low[START+TD:sl_ts+TD]
        trade_start = START+TD
        trade_end = sl_ts
        target_pips = entry - tp
        sl_pips = entry - (entry + df[range_type] * sl_pct_range) 
        if tp_window.min() <= tp:
            if trade_start == trade_end:
                if stop is False:
                    win = 1
                    gain = entry - tp
                else:
                    loss = 1
                    gain = sl_pips
            else:
                win = 1
                gain = entry - tp                                
        else:
            if stop is True:
                loss = 1
                gain = sl_pips
            else:
                gain = entry - close_window.iloc[-1] if close_window.empty is False else 0
                if gain > 0:
                    win = 1
                else:
                    loss = 1
    
    data = win, loss, \
        target_pips, sl_pips, \
        gain, \
        trade_start, trade_end
    
    return data


### Simulate Limit Long Position

In [417]:
def limit_long_gains(
        df: Series, 
        high: Series, 
        low: Series,
        close: Series,
        signal_name: str,
        sl_pct_range: int,
        tp_pct_range: int,
        limit_entry: Series,
        range_type: str = "ADR",
        s_r: bool = False
        ):
    """Get the pip gain and apply to df"""

    win = 0
    loss = 0
    target_pips = 0 
    sl_pips = 0
    gain = 0
    trade_start =  None
    trade_end = None

    signal = None
    if s_r is True:
        if df["S_R_Signal"] in [3]:
            signal = True
        elif df["S_R_Signal"] in [1,2,4]:
            signal = False
    else:
        signal = df[signal_name]
    
    if signal is True:
        idx = int(df["Idx"])
        iday_idx = int(df["Iday_Idx"])
        entry = limit_entry.iloc[idx] 
        # get signal start of fx day timestamp
        day_start_ts = low.iloc[idx-iday_idx:idx-iday_idx+1].index[0]
        OFFSET = day_start_ts + pd.DateOffset(days=1) # end of signal's fx day
        START = df.name # signal start time
        TD = pd.Timedelta(minutes=15)
        END = OFFSET - TD # 17:00 on the signals fx day

        # get stop loss time:
        sl_window = low.loc[START+TD:END] # from signal idx+1 to EOD
        close_window = close.loc[START+TD:END] # from signal idx+1 to EOD
        stop = False # update if stopped 
        sl_ts = None # get timestamp when stopped out

        for i in range(len(sl_window)):
            stop_condition = sl_window.iloc[i] <= (entry - df[range_type] * sl_pct_range)
            if stop_condition:
                stop = True # trade hit stop loss
                # get stop loss timestamp
                sl_ts = sl_window.iloc[i:i+1].index[0] 
                break

        # if not stopped out then the stop window is signal:EOD
        # don't include trade if it is the last candle of the day
        if stop is False:
            if close_window.empty is False:
                sl_ts = close_window.iloc[-1:].index[0]
            else:
                sl_ts = START+TD

        # Take profit price
        tp = entry + (df[range_type] * tp_pct_range)
           
        # trade window
        tp_window = high.loc[START+TD:sl_ts+TD]
        trade_start = START+TD
        trade_end = sl_ts
        target_pips = tp - entry
        sl_pips = (entry - df[range_type] * sl_pct_range) - entry
        if tp_window.max() >= tp:
            if trade_start == trade_end:
                if stop is False:
                    win = 1
                    gain = tp - entry
                else:
                    loss = 1
                    gain = sl_pips
            else:
                win = 1
                gain = tp - entry
        else:
            if stop is True:
                loss = 1
                gain = sl_pips
            else:
                gain = close_window.iloc[-1] - entry if close_window.empty is False else 0
                if gain > 0:
                    win = 1
                else:
                    loss = 1
    
    data = win, loss, \
        target_pips, sl_pips, \
        gain, \
        trade_start, trade_end, \
        
    return data


### Limit-Based Gains (Long/Short)

In [407]:
def get_limit_gains(df: DataFrame, long: str, short: str, sl_pct_r, tp_pct_r):
    long_df = df.copy() 
    short_df = df.copy()
    # long
    long_df[gains_cols] = long_df.apply(
        limit_long_gains,
        axis=1,
        args=[long_df["High"],long_df["Low"],long_df["Close"],
            long, sl_pct_r, tp_pct_r, long_df["S_R_Entry"]],
        result_type='expand',
        range_type="ATR4",
        s_r = True
    )
    # short
    short_df[gains_cols] = short_df.apply(
        limit_short_gains,
        axis=1,
        args=[short_df["High"],short_df["Low"],short_df["Close"],
            short, sl_pct_r, tp_pct_r, short_df["S_R_Entry"]],
        result_type='expand',
        range_type="ATR4",
        s_r = True
    )
    return pd.concat([long_df, short_df])

### Range-Based Gains (Long/Short)

In [1990]:
def get_range_signal_gains(df: DataFrame, long: str, short: str, sl_pct_r, tp_pct_r):
    long_df = df.copy() 
    short_df = df.copy()
    # long
    long_df[gains_cols] = long_df.apply(
        range_long_gains,
        axis=1,
        args=[long_df["High"],long_df["Low"],long_df["Close"],
            long, sl_pct_r, tp_pct_r, long_df["BB_Upper_16_2"], long_df["ATR4"]],
        result_type='expand',
        range_type="ATR4",
        momentum_trade_mgmt = True
    )
    # short
    short_df[gains_cols] = short_df.apply(
        range_short_gains,
        axis=1,
        args=[short_df["High"],short_df["Low"],short_df["Close"],
            short, sl_pct_r, tp_pct_r, short_df["BB_Lower_16_2"], short_df["ATR4"]],
        result_type='expand',
        range_type="ATR4",
        momentum_trade_mgmt = True
    )
    return pd.concat([long_df, short_df])

### Simulate Long/Short Trend Position

In [9]:
def trend_gains(
        df: Series, 
        close: Series,
        sma: Series,
        long_signal: str,
        short_signal: str
        ):
    """Get the pip gain and apply to df
    
    - long_signal: name of buy signal
    - short_signal: name of sell signal
    """

    win = 0
    loss = 0
    target_pips = 0 
    sl_pips = 0
    gain = 0
    trade_start =  None
    trade_end = None

    if df[long_signal] is True or df[short_signal] is True:
        idx = int(df["Idx"])
        iday_idx = int(df["Iday_Idx"])
        # get signal start of fx day timestamp
        day_start_ts = close.iloc[idx-iday_idx:idx-iday_idx+1].index[0]
        OFFSET = day_start_ts + pd.DateOffset(days=1) # end of signal's fx day
        START = df.name # signal start time
        TD = pd.Timedelta(minutes=15)
        END = OFFSET - TD # 17:00 on the signals fx day

        sma_window = sma.loc[START+TD:END] # from signal idx+1 to EOD
        close_window = close.loc[START+TD:END] # from signal idx+1 to EOD
        stop = False # update if stopped 
        sl_ts = None # get timestamp when stopped out
        
        # find exit timestamp:
        for i in range(len(close_window)): 
            # exit condition
            if df[long_signal] is True:
                exit_condition = 1 if close_window.iloc[i] < sma_window.iloc[i] else 0
            elif df[short_signal] is True:
                exit_condition = 1 if close_window.iloc[i] > sma_window.iloc[i] else 0
            # get stop loss time:
            if exit_condition == 1:
                stop = True # trade hit stop loss
                # get stop loss timestamp
                sl_ts = close_window.iloc[i:i+1].index[0] 
                break
        # if not stopped out then the stop window is signal:EOD
        # don't include trade if it is the last candle of the day
        if stop is False:
            if close_window.empty is False:
                sl_ts = close_window.iloc[-1:].index[0]
            else:
                sl_ts = START+TD if (START+TD) < END else START

        # Win / Loss / Gain / Pips
        if df[long_signal] is True:
            gain = close[sl_ts] - df["Close"]
        elif df[short_signal] is True:
            gain = df["Close"] - close[sl_ts]
        trade_start = START+TD
        trade_end = sl_ts
        target_pips = df["ATR4"]
        sl_pips = -(df["ATR4"])
        win = 1 if gain > 0 else 0
        loss = 1 if gain < 0 else 0
    
    data = win, loss, \
        target_pips, sl_pips, \
        gain, \
        trade_start, trade_end, \
        
    return data



### Trade Stats

In [10]:
def trade_stats(df: DataFrame, signal_name: str):
    win_count = df.query(f"{signal_name} == True and Win > 0")[f"{signal_name}"].count()
    loss_count = df.query(f"{signal_name} == True and Loss > 0")[f"{signal_name}"].count()
    total_trades = win_count + loss_count
    win_rate = win_count/total_trades * 100
    win = df.query(f"{signal_name} == True and Gain > 0")["Gain"]
    loss = df.query(f"{signal_name} == True and Gain < 0")["Gain"]
    win_avg_pips = win.mean()
    loss_avg_pips = loss.mean()
    win_pips = win.sum()
    loss_pips = loss.sum()
    total_pips = win_pips + loss_pips

    stats = {
        "Symbol": df["Symbol"].iloc[0],
        "Start": df.index.min(),
        "End": df.index.max(),
        "Win_Count": win_count,
        "Loss_Count": loss_count,
        "Total_Trades": total_trades,
        "Win_Rate": win_rate,
        "Avg_Win": win_avg_pips,
        "Avg_Loss": loss_avg_pips,
        "Win_Pips": win_pips,
        "Loss_Pips": loss_pips,
        "Total_Pips": total_pips
    }

    return stats

def signal_stats(df: DataFrame, signals: list["str"]):
    stats = [trade_stats(df, signal) for signal in signals]
    return stats

def get_trend_signal_stats(df: DataFrame, long: str, short: str, sma: str = "SMA4"):
    # Apply gains to df
    df[gains_cols] = df.apply(
        trend_gains,
        axis=1,
        args=[df["Close"], df[sma], long, short],
        result_type='expand'
    )

    # get stats for long / short signals
    gain_stats = signal_stats(df, [long,short])

    # build stats dataframe
    return pd.DataFrame(data=gain_stats,
                index=[long,short]
                )

## Price Data Files 

In [2031]:
price_data_files = os.listdir(f"{os.getcwd()}/price_data")

## Breakout

In [12]:
pd.options.display.max_rows = 100
data_end_date: str = "20250311"
path: str = f"FE_GBPUSD_15mins_1yr_End_{data_end_date}.csv"
signals_df = get_fe_price_data(filename=path)

gain_stats_df = get_trend_signal_stats(signals_df, "BBU_BO", "BBL_BO")
gain_stats_df



/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_875/1547086338.py:10: DtypeWarning: Columns (35,46,47,49,50,51,52,57,58,59,60,61,62,63,64,66,67,68,69,77) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{PATH}/{filename}")


,Symbol,Start,End,Win_Count,Loss_Count,Total_Trades,Win_Rate,Avg_Win,Avg_Loss,Win_Pips,Loss_Pips,Total_Pips
BBU_BO,GBPUSD,2024-03-12 17:15:00-04:00,2025-03-12 16:45:00-04:00,167,315,482,34.647303,0.001191,-0.000723,0.198855,-0.227665,-0.028810
BBL_BO,GBPUSD,2024-03-12 17:15:00-04:00,2025-03-12 16:45:00-04:00,170,291,461,36.876356,0.001229,-0.000664,0.208915,-0.193270,0.015645


### Results (first 20)

In [13]:
signals_df[gains_cols].query("Win > 0 or Loss > 0").iloc[0:20]

,Win,Loss,TP,SL,Gain,Trade_Start,Trade_End
Date,,,,,,,
2024-03-13 03:15:00-04:00,0.0,1.0,0.000806,-0.000806,-0.000300,2024-03-13 03:30:00-04:00,2024-03-13 03:45:00-04:00
2024-03-13 23:30:00-04:00,0.0,1.0,0.000467,-0.000467,-0.000005,2024-03-13 23:45:00-04:00,2024-03-14 00:30:00-04:00
2024-03-14 03:00:00-04:00,1.0,0.0,0.000471,-0.000471,0.000860,2024-03-14 03:15:00-04:00,2024-03-14 04:45:00-04:00
2024-03-14 04:00:00-04:00,0.0,1.0,0.000696,-0.000696,-0.000190,2024-03-14 04:15:00-04:00,2024-03-14 04:45:00-04:00
2024-03-14 08:45:00-04:00,1.0,0.0,0.001711,-0.001711,0.004795,2024-03-14 09:00:00-04:00,2024-03-14 11:45:00-04:00
2024-03-14 09:15:00-04:00,1.0,0.0,0.001802,-0.001802,0.003310,2024-03-14 09:30:00-04:00,2024-03-14 11:45:00-04:00
2024-03-14 20:00:00-04:00,1.0,0.0,0.000471,-0.000471,0.000180,2024-03-14 20:15:00-04:00,2024-03-14 20:45:00-04:00
2024-03-14 20:15:00-04:00,0.0,1.0,0.000546,-0.000546,-0.000185,2024-03-14 20:30:00-04:00,2024-03-14 20:45:00-04:00
2024-03-15 05:30:00-04:00,1.0,0.0,0.000781,-0.000781,0.000440,2024-03-15 05:45:00-04:00,2024-03-15 06:30:00-04:00


## SMA Breakout

In [14]:
pd.options.display.max_rows = 100
data_end_date: str = "20250311"
path: str = f"FE_GBPUSD_15mins_1yr_End_{data_end_date}.csv"
signals_df = get_fe_price_data(filename=path)

gain_stats_df = get_trend_signal_stats(signals_df, "Bull_SMA_BO", "Bear_SMA_BO")
gain_stats_df

/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_875/1547086338.py:10: DtypeWarning: Columns (35,46,47,49,50,51,52,57,58,59,60,61,62,63,64,66,67,68,69,77) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{PATH}/{filename}")


,Symbol,Start,End,Win_Count,Loss_Count,Total_Trades,Win_Rate,Avg_Win,Avg_Loss,Win_Pips,Loss_Pips,Total_Pips
Bull_SMA_BO,GBPUSD,2024-03-12 17:15:00-04:00,2025-03-12 16:45:00-04:00,62,72,134,46.268657,0.001410,-0.000623,0.087405,-0.044855,0.04255
Bear_SMA_BO,GBPUSD,2024-03-12 17:15:00-04:00,2025-03-12 16:45:00-04:00,57,95,152,37.500000,0.001299,-0.000639,0.074015,-0.060665,0.01335


### Results (first 20)

In [15]:
signals_df[gains_cols].query("Win > 0 or Loss > 0").iloc[0:20]

,Win,Loss,TP,SL,Gain,Trade_Start,Trade_End
Date,,,,,,,
2024-03-13 14:45:00-04:00,0.0,1.0,0.000566,-0.000566,-0.000105,2024-03-13 15:00:00-04:00,2024-03-13 15:30:00-04:00
2024-03-13 20:00:00-04:00,0.0,1.0,0.000385,-0.000385,-0.000275,2024-03-13 20:15:00-04:00,2024-03-13 20:45:00-04:00
2024-03-15 03:45:00-04:00,0.0,1.0,0.000676,-0.000676,-0.000635,2024-03-15 04:00:00-04:00,2024-03-15 04:15:00-04:00
2024-03-15 08:30:00-04:00,0.0,1.0,0.000829,-0.000829,-0.001055,2024-03-15 08:45:00-04:00,2024-03-15 09:00:00-04:00
2024-03-17 23:30:00-04:00,1.0,0.0,0.000300,-0.000300,0.000070,2024-03-17 23:45:00-04:00,2024-03-18 00:15:00-04:00
2024-03-18 01:15:00-04:00,1.0,0.0,0.000276,-0.000276,0.000155,2024-03-18 01:30:00-04:00,2024-03-18 02:45:00-04:00
2024-03-18 08:30:00-04:00,0.0,1.0,0.000606,-0.000606,-0.000315,2024-03-18 08:45:00-04:00,2024-03-18 09:15:00-04:00
2024-03-18 23:30:00-04:00,0.0,1.0,0.000535,-0.000535,-0.000260,2024-03-18 23:45:00-04:00,2024-03-19 00:00:00-04:00
2024-03-20 00:45:00-04:00,0.0,1.0,0.000341,-0.000341,-0.000275,2024-03-20 01:00:00-04:00,2024-03-20 01:30:00-04:00


## Bollinger Band Reversals

In [19]:
pd.options.display.max_rows = 100
data_end_date: str = "20250311"
path: str = f"FE_GBPUSD_15mins_1yr_End_{data_end_date}.csv"
bull_bbr_df = get_fe_price_data(filename=path)
bear_bbr_df = get_fe_price_data(filename=path)

long_signal = "Bull_BBR_V2"
short_signal = "Bear_BBR_V2"

# long
bull_bbr_df[gains_cols] = bull_bbr_df.apply(
    range_long_gains,
    axis=1,
    args=[bull_bbr_df["High"],bull_bbr_df["Low"],bull_bbr_df["Close"], 
          bull_bbr_df["SMA4_Slope"] ,long_signal, 1.5, 2.5, bull_bbr_df["BB_Upper_16_2"]],
    result_type='expand',
    range_type="ATR4"
)
# short
bear_bbr_df[gains_cols] = bear_bbr_df.apply(
    range_short_gains,
    axis=1,
    args=[bear_bbr_df["High"],bear_bbr_df["Low"],bear_bbr_df["Close"], 
          bear_bbr_df["SMA4_Slope"], short_signal, 1.5, 2.5, bear_bbr_df["BB_Lower_16_2"]],
    result_type='expand',
    range_type="ATR4"
)

bull_bbr_gains_df = trade_stats(bull_bbr_df, long_signal)
bear_bbr_gains_df = trade_stats(bear_bbr_df, short_signal)

bbr_gains_df = pd.DataFrame(data=[bull_bbr_gains_df, bear_bbr_gains_df],
             index=[long_signal,short_signal]
             )

bbr_gains_df

/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_875/1547086338.py:10: DtypeWarning: Columns (35,46,47,49,50,51,52,57,58,59,60,61,62,63,64,66,67,68,69,77) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{PATH}/{filename}")
/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_875/1547086338.py:10: DtypeWarning: Columns (35,46,47,49,50,51,52,57,58,59,60,61,62,63,64,66,67,68,69,77) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{PATH}/{filename}")


,Symbol,Start,End,Win_Count,Loss_Count,Total_Trades,Win_Rate,Avg_Win,Avg_Loss,Win_Pips,Loss_Pips,Total_Pips
Bull_BBR_V2,GBPUSD,2024-03-12 17:15:00-04:00,2025-03-12 16:45:00-04:00,317,339,656,48.323171,0.001916,-0.001285,0.607337,-0.434228,0.173109
Bear_BBR_V2,GBPUSD,2024-03-12 17:15:00-04:00,2025-03-12 16:45:00-04:00,284,357,641,44.305772,0.001929,-0.001309,0.547799,-0.465982,0.081817


In [21]:
bull_bbr_df[[*gains_cols, "Bull_BBR_C1","Bull_BBR_C2","Bull_BBR_C3","Bull_BBR_C4"]].nlargest(10, "Gain")

,Win,Loss,TP,SL,Gain,Trade_Start,Trade_End,Bull_BBR_C1,Bull_BBR_C2,Bull_BBR_C3,Bull_BBR_C4
Date,,,,,,,,,,,
2024-08-05 02:15:00-04:00,1.0,0.0,0.007772,-0.004663,0.007772,2024-08-05 02:30:00-04:00,2024-08-05 16:45:00-04:00,NaN,NaN,True,NaN
2025-02-12 09:00:00-05:00,1.0,0.0,0.007403,-0.004442,0.007403,2025-02-12 09:15:00-05:00,2025-02-12 12:15:00-05:00,NaN,NaN,True,NaN
2025-02-12 08:45:00-05:00,1.0,0.0,0.007147,-0.004288,0.007147,2025-02-12 09:00:00-05:00,2025-02-12 12:15:00-05:00,NaN,NaN,True,NaN
2024-08-05 02:00:00-04:00,1.0,0.0,0.006412,-0.003847,0.006412,2024-08-05 02:15:00-04:00,2024-08-05 16:45:00-04:00,NaN,NaN,True,NaN
2025-01-17 02:45:00-05:00,1.0,0.0,0.005366,-0.003219,0.005366,2025-01-17 03:00:00-05:00,2025-01-17 10:00:00-05:00,True,NaN,NaN,NaN
2025-02-06 07:45:00-05:00,1.0,0.0,0.007922,-0.004753,0.005365,2025-02-06 08:00:00-05:00,2025-02-06 11:00:00-05:00,NaN,NaN,NaN,True
2025-01-17 02:30:00-05:00,1.0,0.0,0.005238,-0.003143,0.005238,2025-01-17 02:45:00-05:00,2025-01-17 10:00:00-05:00,NaN,NaN,True,NaN
2024-08-15 08:45:00-04:00,1.0,0.0,0.005203,-0.003122,0.005203,2024-08-15 09:00:00-04:00,2024-08-15 16:45:00-04:00,NaN,NaN,True,NaN
2024-11-06 01:45:00-05:00,1.0,0.0,0.005100,-0.003060,0.005100,2024-11-06 02:00:00-05:00,2024-11-06 10:15:00-05:00,True,NaN,NaN,True


## Breakout Momentum

In [22]:
pd.options.display.max_rows = 100
data_end_date: str = "20250311"
path: str = f"FE_GBPUSD_15mins_1yr_End_{data_end_date}.csv"
bom_df = get_fe_price_data(filename=path)

long_signal = "Bull_BM"
short_signal = "Bear_BM"

bom_gains_df = get_range_signal_gains(bom_df, long_signal, short_signal, 1, 1.25)
bom_stats = signal_stats(bom_gains_df, [long_signal,short_signal])
bom_stats_df = pd.DataFrame(data=bom_stats, index=[long_signal,short_signal])
bom_stats_df

/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_875/1547086338.py:10: DtypeWarning: Columns (35,46,47,49,50,51,52,57,58,59,60,61,62,63,64,66,67,68,69,77) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{PATH}/{filename}")


,Symbol,Start,End,Win_Count,Loss_Count,Total_Trades,Win_Rate,Avg_Win,Avg_Loss,Win_Pips,Loss_Pips,Total_Pips
Bull_BM,GBPUSD,2024-03-12 17:15:00-04:00,2025-03-12 16:45:00-04:00,380,403,783,48.531290,0.001039,-0.000927,0.394732,-0.372677,0.022054
Bear_BM,GBPUSD,2024-03-12 17:15:00-04:00,2025-03-12 16:45:00-04:00,275,314,589,46.689304,0.001236,-0.000956,0.339808,-0.298172,0.041636


### Results (first 20)

In [23]:
bom_gains_df
bom_gains_df[gains_cols].query("Win > 0 or Loss > 0").iloc[0:20]

,Win,Loss,TP,SL,Gain,Trade_Start,Trade_End
Date,,,,,,,
2024-03-14 03:30:00-04:00,1.0,0.0,0.000823,-0.000659,0.000823,2024-03-14 03:45:00-04:00,2024-03-14 08:15:00-04:00
2024-03-14 04:30:00-04:00,1.0,0.0,0.000992,-0.000794,0.000992,2024-03-14 04:45:00-04:00,2024-03-14 08:15:00-04:00
2024-03-18 01:30:00-04:00,1.0,0.0,0.000353,-0.000282,0.000353,2024-03-18 01:45:00-04:00,2024-03-18 06:30:00-04:00
2024-03-18 01:45:00-04:00,1.0,0.0,0.000339,-0.000271,0.000339,2024-03-18 02:00:00-04:00,2024-03-18 06:15:00-04:00
2024-03-18 02:00:00-04:00,0.0,1.0,0.000394,-0.000315,-0.000315,2024-03-18 02:15:00-04:00,2024-03-18 02:45:00-04:00
2024-03-18 02:15:00-04:00,1.0,0.0,0.000370,-0.000296,0.000370,2024-03-18 02:30:00-04:00,2024-03-18 06:15:00-04:00
2024-03-18 02:30:00-04:00,0.0,1.0,0.000342,-0.000274,-0.000274,2024-03-18 02:45:00-04:00,2024-03-18 02:45:00-04:00
2024-03-18 04:15:00-04:00,0.0,1.0,0.000727,-0.000581,-0.000581,2024-03-18 04:30:00-04:00,2024-03-18 05:30:00-04:00
2024-03-18 04:30:00-04:00,0.0,1.0,0.000602,-0.000481,-0.000481,2024-03-18 04:45:00-04:00,2024-03-18 05:30:00-04:00


# GBP

In [2033]:
gbp_data = [x for x in price_data_files if "GBPUSD" in x]
eur_data = [x for x in price_data_files if "EUR" in x]
cad_data = [x for x in price_data_files if "CAD" in x]
jpy_data = [x for x in price_data_files if "JPY" in x]
aud_data = [x for x in price_data_files if "AUD" in x]
nzd_data = [x for x in price_data_files if "NZD" in x]
chf_data = [x for x in price_data_files if "CHF" in x]
latest = [x for x in price_data_files if "latest" in x]
gbp_aud_data = [x for x in price_data_files if "GBPAUD" in x]


In [2042]:
def range_based_multi_year_stats(
        df_list: list[DataFrame], 
        long: str, 
        short: str, 
        sl_pct_r: float, 
        tp_pct_r: float,
        get_signal_gains_func: function
        ):
    signal_stats_list = []
    gains_df_list = []
    for df in df_list:
        signal_gains_df: DataFrame = get_signal_gains_func(df, long, short, sl_pct_r, tp_pct_r)
        gains_df_list.append(signal_gains_df)
        long_short_stats = signal_stats(signal_gains_df.between_time("02:00","11:00"), [long,short])
        stats_df = pd.DataFrame(data=long_short_stats, index=[long,short])
        signal_stats_list.append(stats_df)
    return pd.concat(signal_stats_list), pd.concat(gains_df_list)

pd.options.display.max_rows = 100
gbp_data.sort()
gbp_data
gbp_df_list = [get_fe_price_data(filename=x) for x in gbp_data]
# eur_df_list = [get_fe_price_data(filename=x) for x in eur_data]
# cad_df_list = [get_fe_price_data(filename=x) for x in cad_data]
# jpy_df_list = [get_fe_price_data(filename=x) for x in jpy_data]
# aud_df_list = [get_fe_price_data(filename=x) for x in aud_data]
# nzd_df_list = [get_fe_price_data(filename=x) for x in nzd_data]
# chf_df_list = [get_fe_price_data(filename=x) for x in chf_data]
# latest_df_list = [get_fe_price_data(filename=x) for x in latest]
# gbp_aud_df_list = [get_fe_price_data(filename=x) for x in gbp_aud_data]

long_signal = "Bull_TC"
short_signal = "Bear_TC"

stats, gains = range_based_multi_year_stats(gbp_df_list, long_signal, short_signal, 1, 1.25, get_range_signal_gains)

/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_875/1547086338.py:10: DtypeWarning: Columns (53,54,67,68,73) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{PATH}/{filename}")
/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_875/1547086338.py:10: DtypeWarning: Columns (51,53,56,61,66,68,72,73) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{PATH}/{filename}")
/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_875/1547086338.py:10: DtypeWarning: Columns (37,49,51,53,54,55,56,61,62,63,64,65,66,67,68,70,71,72,73,81) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{PATH}/{filename}")
/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_875/1547086338.py:10: DtypeWarning: Columns (49,50,51,52,53,54,55,56,61,62,63,64,65,66,67,68,70,71,72,73,81) have mixed types. Specify dtype option on import or set low_memory=False.

## Results (5 Years)

In [2043]:
g = gains.sort_index()
stats
# stats["Total_Pips"].sum()

,Symbol,Start,End,Win_Count,Loss_Count,Total_Trades,Win_Rate,Avg_Win,Avg_Loss,Win_Pips,Loss_Pips,Total_Pips
Bull_TC,GBPUSD,2021-03-12 02:00:00-05:00,2022-03-11 11:00:00-05:00,245,297,542,45.202952,0.001399,-0.001097,0.342775,-0.325889,0.016886
Bear_TC,GBPUSD,2021-03-12 02:00:00-05:00,2022-03-11 11:00:00-05:00,276,262,538,51.301115,0.001402,-0.001174,0.386894,-0.307601,0.079293
Bull_TC,GBPUSD,2022-03-11 02:00:00-05:00,2023-03-10 11:00:00-05:00,289,361,650,44.461538,0.002075,-0.001699,0.599648,-0.613301,-0.013653
Bear_TC,GBPUSD,2022-03-11 02:00:00-05:00,2023-03-10 11:00:00-05:00,342,341,683,50.073206,0.002014,-0.001731,0.688782,-0.590399,0.098383
Bull_TC,GBPUSD,2023-03-14 02:00:00-04:00,2024-03-12 11:00:00-04:00,275,276,551,49.909256,0.001427,-0.001163,0.392337,-0.319742,0.072594
Bear_TC,GBPUSD,2023-03-14 02:00:00-04:00,2024-03-12 11:00:00-04:00,252,264,516,48.837209,0.001369,-0.001211,0.344992,-0.314864,0.030128
Bull_TC,GBPUSD,2024-03-13 02:00:00-04:00,2025-03-12 11:00:00-04:00,264,260,524,50.381679,0.001148,-0.000941,0.303150,-0.244609,0.058542
Bear_TC,GBPUSD,2024-03-13 02:00:00-04:00,2025-03-12 11:00:00-04:00,200,233,433,46.189376,0.001206,-0.001025,0.241147,-0.238781,0.002365
Bull_TC,GBPUSD,2025-03-12 02:00:00-04:00,2026-03-11 11:00:00-04:00,268,321,589,45.500849,0.001373,-0.001089,0.367972,-0.349415,0.018557
Bear_TC,GBPUSD,2025-03-12 02:00:00-04:00,2026-03-11 11:00:00-04:00,261,278,539,48.423006,0.001319,-0.001071,0.344319,-0.297784,0.046536


### Trades (Last 50)

In [1995]:
g[[*gains_cols, "ATR4", "SMA4_Slope", "SMA16_Slope", "SMA32_Slope"]].query(
    "Win > 0 or Loss > 0"
).between_time("02:00","11:00").tail(30)

,Win,Loss,TP,SL,Gain,Trade_Start,Trade_End,ATR4,SMA4_Slope,SMA16_Slope,SMA32_Slope
Date,,,,,,,,,,,
2026-03-04 10:45:00-05:00,1.0,0.0,0.002025,-0.001620,0.001505,2026-03-04 11:00:00-05:00,2026-03-04 12:15:00-05:00,0.001620,18.863588,-22.058201,6.359843
2026-03-05 02:45:00-05:00,1.0,0.0,0.001730,-0.001384,0.001730,2026-03-05 03:00:00-05:00,2026-03-05 04:00:00-05:00,0.001384,72.751547,12.053992,-25.456395
2026-03-05 03:00:00-05:00,0.0,1.0,0.002116,-0.001692,-0.001692,2026-03-05 03:15:00-05:00,2026-03-05 04:00:00-05:00,0.001692,-6.889837,-18.863588,-37.812953
2026-03-05 05:30:00-05:00,0.0,1.0,0.001609,-0.001288,-0.001288,2026-03-05 05:45:00-05:00,2026-03-05 05:45:00-05:00,0.001288,-61.498640,43.056947,-23.151754
2026-03-05 05:45:00-05:00,1.0,0.0,0.001689,-0.001351,0.001689,2026-03-05 06:00:00-05:00,2026-03-05 09:30:00-05:00,0.001351,-56.601512,32.262169,-24.028906
2026-03-05 09:00:00-05:00,1.0,0.0,0.001744,-0.001395,0.001130,2026-03-05 09:15:00-05:00,2026-03-05 09:30:00-05:00,0.001395,-73.667289,-16.370151,35.785320
2026-03-06 03:00:00-05:00,0.0,1.0,0.001092,-0.000874,-0.000874,2026-03-06 03:15:00-05:00,2026-03-06 04:00:00-05:00,0.000874,28.442929,-4.051684,13.190611
2026-03-06 07:45:00-05:00,0.0,1.0,0.001792,-0.001434,-0.001434,2026-03-06 08:00:00-05:00,2026-03-06 08:00:00-05:00,0.001434,57.859993,-57.876877,-36.078796
2026-03-06 08:45:00-05:00,1.0,0.0,0.003556,-0.002845,0.003556,2026-03-06 09:00:00-05:00,2026-03-06 10:00:00-05:00,0.002845,77.242468,52.073538,-4.111065


### Trades (1 Day)

In [2000]:
x = g['2026-03-11 02:00:00-04:00':'2026-03-11 11:00:00-04:00'][[*gains_cols, "Close_Pct_DHigh", "Iday_Range", "ADR"]].query(
    "Win > 0 or Loss > 0"
).between_time("02:00","11:00")
x.query("Gain > 0")["Gain"].sum() + x.query("Gain < 0")["Gain"].sum()
x

,Win,Loss,TP,SL,Gain,Trade_Start,Trade_End,Close_Pct_DHigh,Iday_Range,ADR
Date,,,,,,,,,,
2026-03-11 02:00:00-04:00,0.0,1.0,0.000759,-0.000607,-0.000607,2026-03-11 02:15:00-04:00,2026-03-11 02:45:00-04:00,0.113804,0.004745,0.009747
2026-03-11 02:15:00-04:00,0.0,1.0,0.000814,-0.000651,-0.000651,2026-03-11 02:30:00-04:00,2026-03-11 02:45:00-04:00,0.129610,0.004745,0.009747
2026-03-11 02:30:00-04:00,0.0,1.0,0.000817,-0.000654,-0.000654,2026-03-11 02:45:00-04:00,2026-03-11 02:45:00-04:00,0.138040,0.004745,0.009747
2026-03-11 03:00:00-04:00,0.0,1.0,0.000984,-0.000787,-0.000787,2026-03-11 03:15:00-04:00,2026-03-11 03:15:00-04:00,0.291886,0.004745,0.009747
2026-03-11 06:15:00-04:00,1.0,0.0,0.001825,-0.001460,0.001825,2026-03-11 06:30:00-04:00,2026-03-11 08:45:00-04:00,0.378938,0.005555,0.009747
2026-03-11 10:30:00-04:00,1.0,0.0,0.002333,-0.001866,0.002333,2026-03-11 10:45:00-04:00,2026-03-11 16:45:00-04:00,0.556962,0.006320,0.009747
2026-03-11 10:45:00-04:00,1.0,0.0,0.002270,-0.001816,0.002270,2026-03-11 11:00:00-04:00,2026-03-11 16:45:00-04:00,0.537975,0.006320,0.009747
2026-03-11 11:00:00-04:00,1.0,0.0,0.001803,-0.001442,0.001803,2026-03-11 11:15:00-04:00,2026-03-11 16:45:00-04:00,0.686709,0.006320,0.009747


In [30]:
def trend_based_multi_year_stats(
        df_list: list[DataFrame], 
        long: str, 
        short: str,
        sma: str
        ):
    signal_stats_list = []
    for df in df_list:
        stats_df = get_trend_signal_stats(df, long, short, sma)
        signal_stats_list.append(stats_df)
    return pd.concat(signal_stats_list)

gbp_df_list = [get_fe_price_data(filename=x) for x in gbp_data]
long_signal = "Bull_SMA_BO"
short_signal = "Bear_SMA_BO"
trend_based_multi_year_stats(gbp_df_list, long_signal, short_signal, "SMA4")

/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_875/1547086338.py:10: DtypeWarning: Columns (49,50,63,64,69) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{PATH}/{filename}")
/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_875/1547086338.py:10: DtypeWarning: Columns (47,49,52,57,62,64,68,69) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{PATH}/{filename}")
/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_875/1547086338.py:10: DtypeWarning: Columns (35,46,47,49,50,51,52,57,58,59,60,61,62,63,64,66,67,68,69,77) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{PATH}/{filename}")
/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_875/1547086338.py:10: DtypeWarning: Columns (46,47,48,49,50,51,52,57,58,59,60,61,62,63,64,66,67,68,69,77) have mixed types. Specify dtype option on import or set low_memory=False.
  

,Symbol,Start,End,Win_Count,Loss_Count,Total_Trades,Win_Rate,Avg_Win,Avg_Loss,Win_Pips,Loss_Pips,Total_Pips
Bull_SMA_BO,GBPUSD,2021-03-11 17:15:00-05:00,2022-03-11 16:45:00-05:00,50,79,129,38.759690,0.001025,-0.000758,0.051270,-0.059890,-0.008620
Bear_SMA_BO,GBPUSD,2021-03-11 17:15:00-05:00,2022-03-11 16:45:00-05:00,66,125,191,34.554974,0.001198,-0.000701,0.079070,-0.087665,-0.008595
Bull_SMA_BO,GBPUSD,2022-03-10 17:15:00-05:00,2023-03-10 16:45:00-05:00,43,95,138,31.159420,0.001617,-0.001127,0.069545,-0.107035,-0.037490
Bear_SMA_BO,GBPUSD,2022-03-10 17:15:00-05:00,2023-03-10 16:45:00-05:00,52,108,160,32.500000,0.001764,-0.001333,0.091715,-0.144005,-0.052290
Bull_SMA_BO,GBPUSD,2023-03-13 17:15:00-04:00,2024-03-12 16:45:00-04:00,56,80,136,41.176471,0.001268,-0.000769,0.071025,-0.061540,0.009485
Bear_SMA_BO,GBPUSD,2023-03-13 17:15:00-04:00,2024-03-12 16:45:00-04:00,63,110,173,36.416185,0.001091,-0.000722,0.068735,-0.079365,-0.010630
Bull_SMA_BO,GBPUSD,2024-03-12 17:15:00-04:00,2025-03-12 16:45:00-04:00,62,72,134,46.268657,0.001410,-0.000623,0.087405,-0.044855,0.042550
Bear_SMA_BO,GBPUSD,2024-03-12 17:15:00-04:00,2025-03-12 16:45:00-04:00,57,95,152,37.500000,0.001299,-0.000639,0.074015,-0.060665,0.013350
Bull_SMA_BO,GBPUSD,2025-03-11 17:15:00-04:00,2026-03-11 16:45:00-04:00,37,81,118,31.355932,0.001079,-0.000836,0.039940,-0.067695,-0.027755
Bear_SMA_BO,GBPUSD,2025-03-11 17:15:00-04:00,2026-03-11 16:45:00-04:00,65,87,152,42.763158,0.001307,-0.000592,0.084975,-0.051520,0.033455


## Limits

In [436]:
gbp_data = [x for x in price_data_files if "GBP" in x]
eur_data = [x for x in price_data_files if "EUR" in x]
cad_data = [x for x in price_data_files if "CAD" in x]
jpy_data = [x for x in price_data_files if "JPY" in x]
aud_data = [x for x in price_data_files if "AUD" in x]
nzd_data = [x for x in price_data_files if "NZD" in x]
chf_data = [x for x in price_data_files if "CHF" in x]
latest = [x for x in price_data_files if "latest" in x]

pd.options.display.max_rows = 100
gbp_data.sort()
gbp_data
gbp_df_list = [get_fe_price_data(filename=x) for x in gbp_data]

long = "S_R_Signal"
short = "S_R_Signal"
#limit_gains = get_limit_gains(gbp_df_list,long, short, 1 ,1.5 )
stats, gains = range_based_multi_year_stats(gbp_df_list, long, short, 1.5, 3, get_limit_gains)
stats

/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_875/1547086338.py:10: DtypeWarning: Columns (49,50,63,64,69) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{PATH}/{filename}")
/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_875/1547086338.py:10: DtypeWarning: Columns (47,49,52,57,62,64,68,69) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{PATH}/{filename}")
/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_875/1547086338.py:10: DtypeWarning: Columns (35,46,47,49,50,51,52,57,58,59,60,61,62,63,64,66,67,68,69,77) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{PATH}/{filename}")
/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_875/1547086338.py:10: DtypeWarning: Columns (46,47,48,49,50,51,52,57,58,59,60,61,62,63,64,66,67,68,69,77) have mixed types. Specify dtype option on import or set low_memory=False.
  

,Symbol,Start,End,Win_Count,Loss_Count,Total_Trades,Win_Rate,Avg_Win,Avg_Loss,Win_Pips,Loss_Pips,Total_Pips
S_R_Signal,GBPUSD,2021-03-12 02:00:00-05:00,2022-03-11 11:00:00-05:00,74,163,237,31.223629,0.002982,-0.001702,0.220702,-0.277477,-0.056775
S_R_Signal,GBPUSD,2021-03-12 02:00:00-05:00,2022-03-11 11:00:00-05:00,74,163,237,31.223629,0.002982,-0.001702,0.220702,-0.277477,-0.056775
S_R_Signal,GBPUSD,2022-03-11 02:00:00-05:00,2023-03-10 11:00:00-05:00,83,133,216,38.425926,0.003805,-0.002541,0.315816,-0.337998,-0.022182
S_R_Signal,GBPUSD,2022-03-11 02:00:00-05:00,2023-03-10 11:00:00-05:00,83,133,216,38.425926,0.003805,-0.002541,0.315816,-0.337998,-0.022182
S_R_Signal,GBPUSD,2023-03-14 02:00:00-04:00,2024-03-12 11:00:00-04:00,72,134,206,34.951456,0.002892,-0.001639,0.208201,-0.214759,-0.006557
S_R_Signal,GBPUSD,2023-03-14 02:00:00-04:00,2024-03-12 11:00:00-04:00,72,134,206,34.951456,0.002892,-0.001639,0.208201,-0.214759,-0.006557
S_R_Signal,GBPUSD,2024-03-13 02:00:00-04:00,2025-03-12 11:00:00-04:00,60,148,208,28.846154,0.002417,-0.001312,0.145034,-0.194147,-0.049113
S_R_Signal,GBPUSD,2024-03-13 02:00:00-04:00,2025-03-12 11:00:00-04:00,60,148,208,28.846154,0.002417,-0.001312,0.145034,-0.194147,-0.049113
S_R_Signal,GBPUSD,2025-03-12 02:00:00-04:00,2026-03-11 11:00:00-04:00,83,116,199,41.708543,0.002744,-0.001428,0.227757,-0.165695,0.062063
S_R_Signal,GBPUSD,2025-03-12 02:00:00-04:00,2026-03-11 11:00:00-04:00,83,116,199,41.708543,0.002744,-0.001428,0.227757,-0.165695,0.062063


In [439]:
x = gains
x[[*gains_cols, "ATR4", "SMA4_Slope", "SMA4_Slope_SMA"]].query(
    "Win > 0 or Loss > 0"
).between_time("02:00","11:00").tail(10)

,Win,Loss,TP,SL,Gain,Trade_Start,Trade_End,ATR4,SMA4_Slope,SMA4_Slope_SMA
Date,,,,,,,,,,
2026-02-24 04:00:00-05:00,0.0,1.0,0.002955,-0.001478,-0.001478,2026-02-24 04:15:00-05:00,2026-02-24 05:30:00-05:00,0.000985,61.389540,48.617818
2026-02-24 09:15:00-05:00,0.0,1.0,0.003424,-0.001712,-0.001712,2026-02-24 09:30:00-05:00,2026-02-24 10:15:00-05:00,0.001141,-14.931417,-6.908451
2026-02-27 03:00:00-05:00,1.0,0.0,0.004819,-0.002409,0.004819,2026-02-27 03:15:00-05:00,2026-02-27 16:45:00-05:00,0.001606,76.821809,35.284009
2026-02-27 04:30:00-05:00,1.0,0.0,0.002640,-0.001320,0.002640,2026-02-27 04:45:00-05:00,2026-02-27 16:45:00-05:00,0.000880,-34.183040,6.722660
2026-03-04 09:30:00-05:00,1.0,0.0,0.004984,-0.002492,0.000260,2026-03-04 09:45:00-05:00,2026-03-04 16:45:00-05:00,0.001661,44.152133,8.314222
2026-03-04 10:15:00-05:00,0.0,1.0,0.005625,-0.002813,-0.001000,2026-03-04 10:30:00-05:00,2026-03-04 16:45:00-05:00,0.001875,-26.373747,0.456762
2026-03-06 10:00:00-05:00,0.0,1.0,0.006709,-0.003354,-0.003354,2026-03-06 10:15:00-05:00,2026-03-06 10:15:00-05:00,0.002236,73.379243,-33.380714
2026-03-09 07:45:00-04:00,0.0,1.0,0.003949,-0.001974,-0.001974,2026-03-09 08:00:00-04:00,2026-03-09 09:00:00-04:00,0.001316,14.931417,40.669845
2026-03-09 10:30:00-04:00,0.0,1.0,0.005531,-0.002766,-0.002766,2026-03-09 10:45:00-04:00,2026-03-09 11:00:00-04:00,0.001844,-62.396020,-3.149456
